# 🧠 การถดถอยลาสโซ (Lasso Regression - L1 Regularization) และการคัดเลือกคุณลักษณะ (Feature Selection)

ยินดีต้อนรับสู่โน้ตบุ๊กประกอบการอธิบายเรื่อง **การถดถอยแบบลาสโซ (Lasso Regression)**! ในโน้ตบุ๊กนี้เราจะ:
1. สร้างชุดข้อมูลจำลองตามกรณีศึกษากล่องระบุตำแหน่ง Bounding Box (ประกอบด้วยคุณลักษณะเชิงทำนายนัยสำคัญ 3 ตัว และสัญญาณรบกวน 3 ตัว)
2. เทรนแบบจำลองการถดถอยปกติ (OLS) เพื่อสังเกตว่ามีการตั้งค่าน้ำหนักที่ไม่เป็นศูนย์ให้กับคุณลักษณะสัญญาณรบกวนอย่างไร
3. เทรนแบบจำลอง Lasso Regression (L1 Regularization) เพื่อดูพฤติกรรมการปรับค่าน้ำหนักสัญญาณรบกวนให้เป็น **ศูนย์แบบสมบูรณ์** (การคัดเลือกคุณลักษณะแบบอัตโนมัติ)
4. เปรียบเทียบกับแบบจำลอง Ridge Regression (L2 Regularization) ที่ลดขนาดค่าน้ำหนักลงแต่ไม่ลดจนเหลือศูนย์
5. ลงมือสร้าง **Lasso Regression จากศูนย์ (from scratch)** โดยใช้อัลกอริทึม **Coordinate Descent** และตัวดำเนินการขีดจำกัดแบบอ่อน **(Soft-Thresholding Operator)**:
   $$w_j \leftarrow \text{soft\_threshold}(\rho_j, \lambda) / z_j$$

เริ่มต้นด้วยการนำเข้าไลบรารีที่จำเป็นกันก่อนครับ

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error

# กำหนดค่า seed เพื่อให้ได้ผลลัพธ์การสุ่มเหมือนเดิมทุกครั้ง
np.random.seed(42)

## 1. การสร้างข้อมูลตามกรณีศึกษา (Case Study Data Generation)

เราจะสร้างชุดข้อมูลจำลองของกล่อง Bounding Box ที่อธิบายด้วยคุณลักษณะต่างๆ ดังนี้:
1.  `width` ความกว้าง (ค่าน้ำหนักจริง = 1.5)
2.  `height` ความสูง (ค่าน้ำหนักจริง = 0.8)
3.  `aspect_ratio` อัตราส่วนภาพ (ค่าน้ำหนักจริง = -1.2)
4.  `pixel_intensity_mean` ค่าเฉลี่ยความสว่างของพิกเซล (สัญญาณรบกวน, ค่าน้ำหนักจริง = 0.0)
5.  `center_x` พิกัดกึ่งกลางแกน X (สัญญาณรบกวน, ค่าน้ำหนักจริง = 0.0)
6.  `center_y` พิกัดกึ่งกลางแกน Y (สัญญาณรบกวน, ค่าน้ำหนักจริง = 0.0)

สมการเป้าหมายจริง: $y = 1.5 \cdot \text{width} + 0.8 \cdot \text{height} - 1.2 \cdot \text{aspect\_ratio} + \epsilon$

In [ ]:
m = 100 # จำนวนตัวอย่างข้อมูล

# สร้างคุณลักษณะสำหรับใช้ทำนายที่มีนัยสำคัญจริง
width = np.random.rand(m, 1) * 4 + 1
height = np.random.rand(m, 1) * 4 + 1
aspect_ratio = width / height

# สร้างคุณลักษณะที่เป็นสัญญาณรบกวน (Noise)
pixel_intensity = np.random.randn(m, 1) * 10 + 128
center_x = np.random.rand(m, 1) * 640
center_y = np.random.rand(m, 1) * 480

# รวมมิติของคุณลักษณะทั้งหมดเข้าเป็นเมทริกซ์ X
X = np.hstack((width, height, aspect_ratio, pixel_intensity, center_x, center_y))
feature_names = ['width', 'height', 'aspect_ratio', 'pixel_intensity', 'center_x', 'center_y']

# ความสัมพันธ์เชิงเส้นจริงที่ตั้งไว้
true_weights = np.array([1.5, 0.8, -1.2, 0.0, 0.0, 0.0])
noise = np.random.randn(m) * 0.2

# คำนวณค่าเป้าหมาย y
y = X @ true_weights + noise

## 2. การเปรียบเทียบระหว่าง OLS, Ridge และ Lasso (ด้วย Scikit-Learn)

ลองนำแบบจำลองการถดถอยทั้ง 3 รูปแบบมาเทรนกับข้อมูลเพื่อเปรียบเทียบว่ามีวิธีจัดการกับคุณลักษณะที่เป็นสัญญาณรบกวนอย่างไร

In [ ]:
# OLS
ols = LinearRegression()
ols.fit(X, y)

# Ridge (การปรับโทษแบบ L2)
ridge = Ridge(alpha=10.0)
ridge.fit(X, y)

# Lasso (การปรับโทษแบบ L1)
lasso = Lasso(alpha=0.5)
lasso.fit(X, y)

# รวบรวมค่าสัมประสิทธิ์น้ำหนักที่เรียนรู้ได้ลงในตารางเปรียบเทียบ
df_coefs = pd.DataFrame({
    'Feature': feature_names,
    'True Weight': true_weights,
    'OLS Learned': ols.coef_,
    'Ridge Learned (L2)': ridge.coef_,
    'Lasso Learned (L1)': lasso.coef_
})

print(df_coefs.to_string(index=False))

สังเกตความแตกต่างดังนี้:
-   **OLS** มีการคำนวณค่าน้ำหนักที่พยายามลู่เข้าหาค่าที่ไม่ใช่ศูนย์ (เช่น `-0.0002` หรือ `0.0003`) ให้กับคุณลักษณะที่เป็นสัญญาณรบกวน (`pixel_intensity`, `center_x`, `center_y`)
-   **Ridge** สามารถหดค่าน้ำหนักสัมประสิทธิ์ลงได้เยอะมาก แต่ก็ยังคงทิ้งค่าที่ไม่ใช่ศูนย์เอาไว้เล็กน้อย
-   **Lasso** สามารถตั้งค่าสัมประสิทธิ์ของคุณลักษณะสัญญาณรบกวนให้กลายเป็น **0.0 แบบสมบูรณ์**!

## 3. การสร้าง Lasso จากศูนย์ด้วยวิธีการ Coordinate Descent (Lasso from Scratch)

เนื่องจากฟังก์ชันค่าสัมบูรณ์ในบทลงโทษแบบ L1 นั้นไม่สามารถหาอนุพันธ์ได้ที่จุด $w=0$ เราจึงไม่สามารถใช้วิธีเกรเดียนต์เดสเซนต์ปกติหรือสมการปกติทางคณิตศาสตร์ได้โดยตรง
ในการแก้ปัญหานี้ เราจึงหันมาใช้วิธีการปรับปรุงพารามิเตอร์ทีละแกนพิกัด หรือเรียกว่า **Coordinate Descent**
สำหรับในแต่ละคุณลักษณะ $j$:
1.  คำนวณเป้าหมายความคลาดเคลื่อนสะสม (Residual) โดยละเว้นคุณลักษณะ $j$:
    $$r_i = y_i - \sum_{k \ne j} w_k x_{ik} - b$$
2.  คำนวณค่าสัมประสิทธิ์การฉายแสง (Projection Coefficient) $\rho_j$:
    $$\rho_j = \sum_{i=1}^m x_{ij} r_i$$
3.  คำนวณตัวปรับค่าบรรทัดฐาน (Normalizing Factor) $z_j$:
    $$z_j = \sum_{i=1}^m x_{ij}^2$$
4.  อัปเดตค่าน้ำหนัก $w_j$ โดยใช้ตัวดำเนินการขีดจำกัดแบบอ่อน **(Soft-Thresholding Operator)**:
    $$w_j = \text{soft\_threshold}(\rho_j, \lambda) / z_j$$
    โดยนิยามคือ:
    $$\text{soft\_threshold}(\rho, \lambda) = \text{sign}(\rho) \max(0, |\rho| - \lambda)$$

มาเริ่มเขียนอัลกอริทึมนี้ด้วยไพทอนกันครับ

In [ ]:
def soft_threshold(rho, lmbda):
    """
    ใช้งานตัวดำเนินการ Soft-Thresholding เพื่อลดค่าน้ำหนักให้เป็นศูนย์เมื่ออยู่ใต้ขีดจำกัด
    """
    if rho > lmbda:
        return rho - lmbda
    elif rho < -lmbda:
        return rho + lmbda
    else:
        return 0.0

def fit_lasso_coordinate_descent(X, y, lmbda, epochs=200):
    m, n = X.shape
    w = np.zeros(n)
    b = 0.0
    
    for epoch in range(epochs):
        # อัปเดตค่าจุดตัดแกน Y (อคติ b) ซึ่งไม่โดนปรับโทษปรับแบบ L1
        b = np.mean(y - X @ w)
        
        # อัปเดตค่าน้ำหนักสัมประสิทธิ์ w_j ทีละคอลัมน์
        for j in range(n):
            # คำนวณค่าทำนายผลโดยไม่คิดรวมคุณลักษณะ j
            w_except_j = w.copy()
            w_except_j[j] = 0.0
            y_pred_except_j = X @ w_except_j + b
            
            # คำนวณผลต่างความคลาดเคลื่อน (Residual)
            r = y - y_pred_except_j
            
            # คำนวณค่าการฉายแสง rho_j และตัวปรับสเกล z_j
            rho_j = np.sum(X[:, j] * r)
            z_j = np.sum(X[:, j] ** 2)
            
            # ปรับปรุงค่าน้ำหนักสัมประสิทธิ์โดยใช้สูตร Soft Thresholding
            w[j] = soft_threshold(rho_j, lmbda) / z_j
            
    return w, b

# เริ่มต้นรันอัลกอริทึม Coordinate Descent
lambda_param = 50.0  # เทียบเท่ากับค่า alpha ใน sklearn (ปรับสเกลตามขนาดข้อมูล)
w_scratch, b_scratch = fit_lasso_coordinate_descent(X, y, lambda_param, epochs=300)

print("Lasso coefficients from Scratch Coordinate Descent:")
for name, weight in zip(feature_names, w_scratch):
    print(f"{name:16s}: {weight:.4f}")
print("Bias/Intercept  :", b_scratch)

## 4. การแสดงแผนผังเส้นทางคัดเลือกคุณลักษณะ (Lasso Path Visualization)

เราลองมาพล็อตกราฟสังเกตพฤติกรรมของค่าสัมประสิทธิ์ในการตอบสนองเมื่อเราเพิ่มค่าพารามิเตอร์โทษปรับ $\lambda$ ให้เข้มงวดขึ้นเรื่อยๆ ซึ่งสัมประสิทธิ์ของแต่ละคุณลักษณะจะทยอยลดลงจนเหลือศูนย์แบบเรียงตัวกันไปทีละหนึ่ง

In [ ]:
alphas_path = np.logspace(-2, 2, 100)
coefs = []

for alpha in alphas_path:
    # ปรับสเกลอัลฟ่าให้เหมาะสมสำหรับเปรียบเทียบกับ Scikit-Learn
    w_p, _ = fit_lasso_coordinate_descent(X, y, alpha * m, epochs=100)
    coefs.append(w_p)

coefs = np.array(coefs)

# พล็อตกราฟสัมประสิทธิ์
plt.figure(figsize=(10, 6))
for i in range(len(feature_names)):
    plt.plot(alphas_path, coefs[:, i], label=feature_names[i], linewidth=2)

plt.xscale('log')
plt.xlabel('Regularization Strength (Alpha)')
plt.ylabel('Coefficients')
plt.title('Lasso Path: How Noise Features Drop to Exactly 0 first')
plt.legend()
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.show()

สังเกตว่ากลุ่มคุณลักษณะที่เป็นสัญญาณรบกวน (`center_x`, `center_y`, `pixel_intensity`) จะมีค่าดิ่งตกเหลือ 0 ในทันทีตั้งแต่ช่วงที่ค่าโทษปรับมีขนาดค่อนข้างน้อย ขณะที่คุณลักษณะที่มีความสัมพันธ์จริงกับพิกัดเป้าหมาย (`aspect_ratio`, `width`, `height`) จะคงตัวต้านทานโทษปรับได้ยาวนานกว่ามาก!

## 💡 ความเชื่อมโยงสู่ Deep Learning และ YOLO
*   **การบีบอัดขนาดของโมเดลและความเบาบาง (Model Compression & Sparsity):** การจัดระเบียบแบบ L1 (L1 Regularization) นิยมนำมาใช้สร้าง **เครือข่ายประสาทเทียมแบบเบาบาง (Sparse Neural Networks)** โดยการเพิ่มบทลงโทษแบบ L1 เข้าไปที่ค่าน้ำหนักเพื่อปรับแต่งให้น้ำหนักจานจำนวนมากมีค่าเป็น 0 แบบสมบูรณ์ จากนั้นอัลกอริทึมการตัดแต่งโครงสร้าง (Pruning Algorithms) จะตัดการเชื่อมต่อระหว่างโหนดที่มีค่าน้ำหนักเป็นศูนย์นี้ออกไปโดยสิ้นเชิง ส่งผลให้ขนาดโมเดลเล็กลง (พารามิเตอร์ลดลง) และเพิ่มความเร็วในขั้นตอนการทำนายผลลัพธ์ (Inference Speed) ซึ่งเป็นตัวแปรสำคัญเพื่อให้แบบจำลองระดับลึกอย่าง YOLO ทำงานตรวจจับได้แบบเรียลไทม์บนอุปกรณ์พกพาขนาดเล็ก (Edge Devices)